# 03 · Ad monetisation

**Question.** How much do ads earn, how well are they delivered, and does early exposure push new players away?
**Answer.** About 38,400 USD of estimated revenue in six weeks, rewarded videos earning twice as much per impression as interstitials. Nothing shows that ads push new players away: at the same level of play, more interstitials go with an equal or slightly higher D1. This is observational; notebook 05 holds the only controlled comparison.
**Method.** Revenue is the ad-network estimate of each impression (table `ads`). Exposure is compared within quartiles of first-day activity (table `day0_q`, `sql/04_ad_exposure_day0.sql`). Intervals are 95% Wilson intervals.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from game_events import config, metrics
from game_events.data import connect_events
from game_events.queries import build_base_tables, build_day0_exposure

con, readable, unreadable = connect_events()
build_base_tables(con)
build_day0_exposure(con)
print(f"{len(readable)} readable files, {len(unreadable)} skipped")

## 1. Revenue field: completeness and format

In [ ]:
con.sql("""
    SELECT COUNT(*)                                                               AS impressions,
           COUNT(*) FILTER (WHERE amount_text IS NULL)                            AS no_revenue_value,
           COUNT(*) FILTER (WHERE amount_text IS NOT NULL AND amount_usd IS NULL) AS unparsable,
           COUNT(*) FILTER (WHERE has_amount_key AND has_value_key)               AS both_keys,
           COUNT(*) FILTER (WHERE amount_text LIKE '%,%' OR amount_text LIKE '%/%') AS non_dot_decimal,
           COUNT(*) FILTER (WHERE amount_usd > 1)                                 AS above_1_usd
    FROM ads
""").show(max_width=200)

Every impression has a parsable revenue once commas and slashes are read as decimal separators (640,762 values use them). 1,737 impressions are above 1 USD: they are kept, and a capped version is used in the A/B test.

### Which key carries the revenue, and does its level change at the switch?

In [ ]:
coverage = con.execute("""
    SELECT day, COUNT(*) AS impressions,
           ROUND(100.0 * COUNT(*) FILTER (WHERE has_value_key)  / COUNT(*), 1) AS pct_value_key,
           ROUND(100.0 * COUNT(*) FILTER (WHERE has_amount_key) / COUNT(*), 1) AS pct_amount_key,
           ROUND(100.0 * COUNT(amount_usd) / COUNT(*), 1)                       AS pct_usable,
           ROUND(AVG(amount_usd) * 1000, 1)                                     AS ecpm_usd
    FROM ads GROUP BY 1 ORDER BY 1
""").df()
coverage

In [ ]:
versions = con.execute("""
    SELECT CAST(ts AS DATE) AS day, user_properties_custom['appVersion'] AS app_version,
           COUNT(DISTINCT user_id) AS players
    FROM events
    WHERE CAST(ts AS DATE) BETWEEN DATE '2025-03-22' AND DATE '2025-04-02'
    GROUP BY 1, 2
""").df()
top_versions = versions.groupby("app_version")["players"].sum().nlargest(6).index
(versions[versions["app_version"].isin(top_versions)]
 .pivot(index="day", columns="app_version", values="players").fillna(0).astype(int))

## 2. Revenue by format and network

In [ ]:
con.sql("""
    SELECT ad_format, COUNT(*) AS impressions, ROUND(SUM(amount_usd), 0) AS revenue_usd,
           ROUND(AVG(amount_usd) * 1000, 1) AS ecpm_usd
    FROM ads GROUP BY 1 ORDER BY 3 DESC
""").show(max_width=200)

con.sql("""
    SELECT ad_source, COUNT(*) AS impressions, ROUND(SUM(amount_usd), 0) AS revenue_usd,
           ROUND(AVG(amount_usd) * 1000, 1) AS ecpm_usd
    FROM ads GROUP BY 1 ORDER BY 3 DESC LIMIT 10
""").show(max_width=200)

## 3. Ad revenue per active player and per day

In [ ]:
per_player = con.execute("""
    WITH dau AS (SELECT day, COUNT(*) AS dau FROM active_days GROUP BY 1),
         rev AS (SELECT day, SUM(amount_usd) AS revenue, COUNT(*) AS impressions FROM ads GROUP BY 1)
    SELECT d.day, d.dau, r.impressions, ROUND(r.revenue, 0) AS revenue_usd,
           ROUND(r.revenue / d.dau, 4)           AS revenue_per_active_player,
           ROUND(1.0 * r.impressions / d.dau, 2) AS impressions_per_active_player
    FROM dau d JOIN rev r USING (day)
    ORDER BY 1
""").df()
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(per_player["day"], per_player["revenue_per_active_player"], color="#2a6f97")
ax.set_ylabel("Ad revenue per active player (USD)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
per_player.describe().round(3).loc[["mean", "min", "max"]]

2025-03-27 is the data gap; 2025-02-28 and 2025-04-10 are partial days.

## 4. Ad delivery

In [ ]:
con.sql("""
    SELECT event_name AS lifecycle, event_properties['state'] AS state, COUNT(*) AS events
    FROM events
    WHERE event_name IN ('interstitialLifecycle', 'rvLifecycle')
    GROUP BY 1, 2 ORDER BY 1, 3 DESC
""").show(max_rows=30, max_width=200)

con.sql("""
    SELECT event_name AS lifecycle,
           ROUND(100.0 * COUNT(*) FILTER (WHERE event_properties['state'] = 'AdLoadFailed')
                 / COUNT(*) FILTER (WHERE event_properties['state'] IN ('AdLoaded', 'AdLoadFailed')), 1) AS load_failure_pct
    FROM events
    WHERE event_name IN ('interstitialLifecycle', 'rvLifecycle')
    GROUP BY 1
""").show(max_width=200)

In [ ]:
con.sql("""
    SELECT event_properties['errorInfo'] AS error, COUNT(*) AS events
    FROM events
    WHERE event_name IN ('interstitialLifecycle', 'rvLifecycle')
      AND event_properties['errorInfo'] <> 'no error'
    GROUP BY 1 ORDER BY 2 DESC LIMIT 8
""").show(max_width=250)

## 5. First-day exposure and next-day return

Population: new players, cohorts of 2025-03-26 and 2025-03-27 excluded (day 0 or day 1 on the data gap).

In [ ]:
con.sql("SELECT COUNT(*) AS players, ROUND(100.0 * AVG(retained_d1::INT), 1) AS d1_pct FROM day0_q").show()
metrics.d1_by_exposure(con, "inter_day0")

Players who play longer see more interstitials and also return more: the same comparison follows within groups of similar activity.

In [ ]:
by_sessions = pd.concat([
    metrics.d1_by_exposure(con, "inter_day0", where=f"sessions_day0 {cond}").assign(day0_sessions=label)
    for label, cond in [("1 session", "= 1"), ("2 sessions", "= 2"), ("3+ sessions", ">= 3")]
], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 3.2))
for label, g in by_sessions.groupby("day0_sessions"):
    g = g[g["players"] >= 200]                      # smaller cells give intervals too wide to read
    ax.errorbar(g["bucket"], g["d1_pct"], yerr=[g["d1_pct"] - g["ci_low"], g["ci_high"] - g["d1_pct"]],
                marker="o", capsize=3, label=label)
ax.set_xlabel("Interstitials seen on the first day")
ax.set_ylabel("D1 retention (%)")
ax.legend()
plt.tight_layout()
plt.show()
by_sessions

Rewarded videos are chosen by the player: they reflect interest rather than annoyance.

In [ ]:
metrics.d1_by_exposure(con, "rewarded_day0")

### Within quartiles of first-day activity

Activity = gameplay events on day 0. Quartile 1 is the least active group.

In [ ]:
by_play = pd.concat([
    metrics.d1_by_exposure(con, "inter_day0", where=f"play_quartile = {q}").assign(play_quartile=q)
    for q in (1, 2, 3, 4)
], ignore_index=True)
by_play

## Findings

- **Where the revenue is stored.** The ad revenue of an impression is under the key `value` until 2025-03-28 (100% of impressions), under both keys on 2025-03-29 (85.3% `value`, 14.7% `amount`) and under `amount` from 2025-03-30 (98.5%, then about 100%). With `COALESCE(amount, value)`, every one of the 1,900,860 impressions has a usable revenue, on all 42 days.
- **Revenue.** About 38,400 USD of estimated ad revenue over six weeks. Rewarded videos bring 22,761 USD from 41% of impressions (eCPM, the revenue per 1,000 impressions, is 29.1 USD); interstitials bring 15,656 USD (eCPM 14.0). By network, AppLovin is the largest (22,136 USD, eCPM 22.5); Mintegral (25.8) and Facebook (25.0) pay more per impression; AdMob (16.2), Liftoff (10.1) and Pangle (7.3) pay less.
- **Revenue per active player.** On average 0.23 USD per active player per day (range 0.17 to 0.35), with 11.5 impressions per active player per day (range 9.0 to 13.1). These are estimates reported by the ad networks, not cash received.
- **A shift in level after the rename.** eCPM is about 27% higher from 2025-03-31 than before 2025-03-29, in both formats (rewarded 27.0 to 34.1 USD, interstitial 12.9 to 16.6 USD). It may be real (new app version 0.21.23, new ad configuration) or due to a change in how the value is measured; the data do not say which. Do not compare revenue before and after as if it were one series.
- **Outliers.** 1,737 impressions are above 1 USD each and add up to about 2,700 USD (about 7% of revenue); a version of the revenue capped at 1 USD per impression is given for the tests.
- **Delivery.** Load failures are 3.9% for rewarded videos and 2.5% for interstitials. The most frequent errors are network ones (host not resolved, connection to `127.0.0.1` or `0.0.0.0`); the latter pattern could come from DNS filtering or ad blockers (not verified).
- **Early exposure and retention.** Among 44,170 new players, D1 rises with first-day interstitials: 23.8% with none, 27.7% with 1 to 2, 34.4% with 3 to 5 and 52.3% with 6 or more. Most of this gradient is play volume: when players are split into four groups by first-day gameplay events, D1 goes from 19.8% to 51.9% between the least and most active groups with no ads at all, while inside a group the difference between none and 6 or more interstitials is about 3 to 7 points (51.9% to 58.5% in the most active group; 37.0% to 42.9% in the third). Rewarded ads show the same pattern before controls (29.0% with none, 50.5% with 6 or more).
- **What this does and does not say.** Nothing in these data shows that ads push new players away: within comparable play levels, more ads go with equal or slightly higher D1. This does not prove ads help retention; exposure still reflects how long a player stays, and some cells are small (2 players with 6 or more ads in the least active group). The only controlled comparison is the experiment of notebook 05.
- **The 2025-03-27 gap is not specific to one app version.** On that day players fall by about 60% in every version (1,355 against about 3,500 in 0.21.21; 9 against about 50 in 0.21.15), so a bug of the new version does not explain it. Version 0.21.23 first appears that day (4 players), which may be a coincidence; the cause remains unknown.

**Limits.**
- Revenue is an estimate reported by the ad networks, not cash received.
- Revenue before and after 2025-03-29 may not be comparable (see above).
- Exposure and retention are both driven by how much a player plays; the controls reduce this effect but do not remove it.
- Cohorts of 2025-03-26 and 2025-03-27 are excluded because of the data gap.